# SimCLR-style contrastive SSL pretraining, ps32

This notebook pretrains a modified 14-channel ResNet-18 encoder with contrastive self-supervised learning on unlabeled conditioning-factor raster patches. It does not use landslide/non-landslide labels, does not fine-tune a supervised classifier, and does not generate susceptibility maps.

This is in-domain transductive SSL pretraining: unlabeled patches are sampled from the whole cleaned study area and SCV holdout clusters are not excluded because no class labels are used during this pretraining stage.


## 1. Purpose and experiment configuration

The goal is to learn geospatial patch representations by pulling together two conservative augmented views of the same raster patch and pushing apart views from different patches with NT-Xent loss.

For geospatial raster patches, augmentations are intentionally conservative and applied consistently across all 14 channels. RGB-style color jitter is not used because the channels represent physical conditioning factors rather than image colors.


In [1]:
from pathlib import Path
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

raster_dir = PROJECT_ROOT / "data" / "processed" / "rasters_cleaned"
unlabeled_index_csv = PROJECT_ROOT / "data" / "processed" / "ssl_unlabeled_indices" / "unlabeled_patch_index_ps32_n20000.csv"

output_root = PROJECT_ROOT / "outputs" / "SSL_contrastive_ps32"
training_log_dir = output_root / "training_logs"
figure_root = PROJECT_ROOT / "figures" / "SSL_contrastive_ps32"
training_curve_dir = figure_root / "training_curves"
checkpoint_dir = PROJECT_ROOT / "checkpoints" / "ssl_pretrained" / "contrastive"

training_log_path = training_log_dir / "contrastive_ps32_training_log.csv"
encoder_best_path = checkpoint_dir / "resnet18_contrastive_ps32_encoder_best.pt"
full_model_best_path = checkpoint_dir / "resnet18_contrastive_ps32_full_model_best.pt"
last_checkpoint_path = checkpoint_dir / "resnet18_contrastive_ps32_last.pt"
loss_curve_path = training_curve_dir / "contrastive_ps32_loss_curve.png"

patch_size = 32
n_unlabeled_patches = 20_000
normalization_sample_size = 5_000
max_nodata_ratio = 0.0
nodata_value = -9999
random_seed = 42
max_attempts = 1_000_000

train_fraction = 0.9
batch_size = 128
num_workers = 0
learning_rate = 1e-4
weight_decay = 1e-4
max_epochs = 50
early_stopping_patience = 10
gradient_clip_norm = 5.0
temperature = 0.2
projection_dim = 128

augmentation_config = {
    "use_hflip": True,
    "use_vflip": True,
    "use_rot90": True,
    "use_crop_resize": True,
    "crop_min_size": 28,
    "crop_max_size": 32,
    "use_gaussian_noise": True,
    "noise_prob": 0.5,
    "noise_std": 0.02,
    "channel_dropout_prob": 0.0,
    "max_channels_to_drop": 2,
}

print("Project root:", PROJECT_ROOT)
print("Raster dir:", raster_dir)
print("Unlabeled index:", unlabeled_index_csv)
print("Encoder checkpoint output:", encoder_best_path)


Project root: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project
Raster dir: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\rasters_cleaned
Unlabeled index: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\data\processed\ssl_unlabeled_indices\unlabeled_patch_index_ps32_n20000.csv
Encoder checkpoint output: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\contrastive\resnet18_contrastive_ps32_encoder_best.pt


## 2. Import packages and project modules


In [2]:
import sys
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

sys.path.insert(0, str(PROJECT_ROOT))

from src.patch_dataset import list_raster_files, audit_raster_alignment
from src.ssl_masked_recon import (
    MaskedReconstructionRasterDataset,
    compute_ssl_channel_stats,
    create_unlabeled_patch_index,
)
from src.ssl_contrastive import ContrastiveRasterPatchDataset, ContrastiveResNet18Model
from src.train_ssl import train_contrastive_model
from src.plotting import plot_ssl_loss_curve
from src.utils import ensure_dir, get_device, set_global_seed, count_trainable_parameters

for directory in [training_log_dir, training_curve_dir, checkpoint_dir, unlabeled_index_csv.parent]:
    ensure_dir(directory)


## 3. Set random seeds and device


In [3]:
set_global_seed(random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = bool(torch.cuda.is_available())
print("Using device:", device)
if torch.cuda.is_available():
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    print("torch.version.cuda:", torch.version.cuda)
else:
    print("WARNING: CUDA is not available; contrastive pretraining will run on CPU.")


Using device: cuda
CUDA GPU: NVIDIA GeForce RTX 4090
torch.version.cuda: 11.8


## 4. Load and audit cleaned rasters


In [4]:
raster_files = list_raster_files(raster_dir)
raster_audit = audit_raster_alignment(raster_files, expected_nodata=nodata_value)
print("Number of raster files:", len(raster_files))
print("Raster shape:", (raster_audit.height, raster_audit.width))
print("Raster CRS:", raster_audit.crs)
print("Raster nodata:", raster_audit.nodata)
print("First raster:", raster_files[0].name)
print("Last raster:", raster_files[-1].name)


Number of raster files: 14
Raster shape: (10071, 9596)
Raster CRS: EPSG:32618
Raster nodata: -9999.0
First raster: aspect_30m.tif
Last raster: twi_dinf_30m.tif


## 5. Load or generate unlabeled patch index


In [5]:
unlabeled_index = create_unlabeled_patch_index(
    raster_dir=raster_dir,
    output_csv=unlabeled_index_csv,
    patch_size=patch_size,
    n_patches=n_unlabeled_patches,
    nodata_value=nodata_value,
    max_nodata_ratio=max_nodata_ratio,
    random_seed=random_seed,
    max_attempts=max_attempts,
    force_regenerate=False,
)
valid_unlabeled = unlabeled_index.loc[unlabeled_index["valid_patch"].astype(bool)].copy()
if len(valid_unlabeled) < n_unlabeled_patches:
    raise RuntimeError(f"Expected at least {n_unlabeled_patches} valid unlabeled patches, found {len(valid_unlabeled)}.")
valid_unlabeled = valid_unlabeled.iloc[:n_unlabeled_patches].copy()
if not np.isclose(valid_unlabeled["nodata_ratio"].to_numpy(dtype="float64"), 0.0).all():
    raise ValueError("Unlabeled patch index contains nonzero nodata_ratio rows.")
print("Number of valid unlabeled patches obtained:", len(valid_unlabeled))
print(valid_unlabeled[["patch_id", "row", "col", "nodata_ratio"]].head())


Number of valid unlabeled patches obtained: 20000
       patch_id   row   col  nodata_ratio
0  U_SSL_000001  2038   916           0.0
1  U_SSL_000002  5168  1241           0.0
2  U_SSL_000003  5491  4256           0.0
3  U_SSL_000004  4538  2189           0.0
4  U_SSL_000005   940  5320           0.0


## 6. Compute contrastive SSL channel normalization statistics


In [6]:
raw_stats_dataset = MaskedReconstructionRasterDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    patch_size=patch_size,
    nodata_value=nodata_value,
    normalize=False,
    return_metadata=False,
)
channel_means, channel_stds = compute_ssl_channel_stats(
    raw_stats_dataset,
    sample_size=normalization_sample_size,
    batch_size=64,
    random_seed=random_seed,
)
raw_stats_dataset.close()
print("Channel means shape:", channel_means.shape)
print("Channel stds shape:", channel_stds.shape)
print("Channel means:", np.round(channel_means, 4))
print("Channel stds:", np.round(channel_stds, 4))
if not np.isfinite(channel_means).all() or not np.isfinite(channel_stds).all():
    raise ValueError("Channel normalization statistics contain NaN or inf.")
if (channel_stds <= 0).any():
    raise ValueError("At least one channel std is nonpositive.")


Channel means shape: (14,)
Channel stds shape: (14,)
Channel means: [1.8172540e+02 1.5900377e+03 1.9745800e+01 1.7168539e+02 1.9745800e+01
 4.2806499e+01 1.0002100e+01 6.5240002e-01 1.7899999e-02 1.7500000e-02
 5.2075802e+01 5.5462999e+00 2.4302999e+01 7.0296001e+00]
Channel stds: [7.987370e+01 6.338770e+01 3.289600e+00 1.632223e+02 3.289600e+00
 2.239570e+01 3.565600e+00 1.664000e-01 1.545000e-01 2.632000e-01
 9.396800e+00 5.474800e+00 1.303064e+02 1.296800e+00]


## 7. Define contrastive Dataset and DataLoaders


In [7]:
contrastive_dataset = ContrastiveRasterPatchDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    patch_size=patch_size,
    nodata_value=nodata_value,
    normalize=True,
    channel_means=channel_means,
    channel_stds=channel_stds,
    return_metadata=False,
    augment=True,
    augmentation_config=augmentation_config,
)

view1, view2 = contrastive_dataset[0]
print("Dataset length:", len(contrastive_dataset))
print("view1.shape:", tuple(view1.shape))
print("view2.shape:", tuple(view2.shape))
print("view1 finite:", bool(torch.isfinite(view1).all()))
print("view2 finite:", bool(torch.isfinite(view2).all()))
print("view1 contains nodata:", bool((view1 == nodata_value).any()))
print("view2 contains nodata:", bool((view2 == nodata_value).any()))
view_difference_seen = any(
    float(torch.mean(torch.abs(contrastive_dataset[i][0] - contrastive_dataset[i][1])).item()) > 1e-6
    for i in range(min(10, len(contrastive_dataset)))
)
print("Two augmented views differ in smoke test:", view_difference_seen)

rng = np.random.default_rng(random_seed)
all_indices = np.arange(len(contrastive_dataset))
rng.shuffle(all_indices)
n_train = int(train_fraction * len(all_indices))
train_indices = all_indices[:n_train].tolist()
val_indices = all_indices[n_train:].tolist()
train_dataset = Subset(contrastive_dataset, train_indices)
val_dataset = Subset(contrastive_dataset, val_indices)

loader_generator = torch.Generator()
loader_generator.manual_seed(random_seed)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    generator=loader_generator,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)
print("Train/val split sizes:", len(train_dataset), len(val_dataset))
print("Batch size:", batch_size)
print("pin_memory:", pin_memory)


Dataset length: 20000
view1.shape: (14, 32, 32)
view2.shape: (14, 32, 32)
view1 finite: True
view2 finite: True
view1 contains nodata: False
view2 contains nodata: False
Two augmented views differ in smoke test: True
Train/val split sizes: 18000 2000
Batch size: 128
pin_memory: True


## 8. Define modified ResNet-18 encoder and projection head


In [8]:
model = ContrastiveResNet18Model(
    in_channels=14,
    projection_dim=projection_dim,
    small_patch_stem=True,
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
print("Model parameter count:", count_trainable_parameters(model))
print("Projection dim:", projection_dim)
print("Temperature:", temperature)
print("Augmentation configuration:", augmentation_config)


Model parameter count: 11504512
Projection dim: 128
Temperature: 0.2
Augmentation configuration: {'use_hflip': True, 'use_vflip': True, 'use_rot90': True, 'use_crop_resize': True, 'crop_min_size': 28, 'crop_max_size': 32, 'use_gaussian_noise': True, 'noise_prob': 0.5, 'noise_std': 0.02, 'channel_dropout_prob': 0.0, 'max_channels_to_drop': 2}


## 9. Define geospatial augmentations and NT-Xent contrastive loss

The `ContrastiveRasterPatchDataset` generates two independent augmented views for each unlabeled patch. The training loop forwards both views through the shared encoder/projection head and optimizes NT-Xent loss at temperature 0.2.


In [9]:
smoke_batch = next(iter(train_loader))
smoke_view1, smoke_view2 = smoke_batch[0].to(device, non_blocking=True), smoke_batch[1].to(device, non_blocking=True)
print("model device:", next(model.parameters()).device)
print("view1.device:", smoke_view1.device)
print("view2.device:", smoke_view2.device)
with torch.no_grad():
    h, z = model(smoke_view1[:4])
print("Feature shape:", tuple(h.shape))
print("Projection shape:", tuple(z.shape))
if torch.cuda.is_available():
    assert next(model.parameters()).is_cuda and smoke_view1.is_cuda and smoke_view2.is_cuda


model device: cuda:0
view1.device: cuda:0
view2.device: cuda:0
Feature shape: (4, 512)
Projection shape: (4, 128)


## 10. Train contrastive SSL model


In [10]:
config = {
    "ssl_task": "contrastive_simclr",
    "patch_size": patch_size,
    "n_unlabeled_patches": n_unlabeled_patches,
    "normalization_sample_size": normalization_sample_size,
    "train_fraction": train_fraction,
    "batch_size": batch_size,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "max_epochs": max_epochs,
    "early_stopping_patience": early_stopping_patience,
    "gradient_clip_norm": gradient_clip_norm,
    "temperature": temperature,
    "projection_dim": projection_dim,
    "random_seed": random_seed,
    "augmentation_config": augmentation_config,
    "raster_dir": str(raster_dir),
    "unlabeled_index_csv": str(unlabeled_index_csv),
}

print("Quality check before training")
print("number of raster files:", len(raster_files))
print("raster shape:", (raster_audit.height, raster_audit.width))
print("patch_size:", patch_size)
print("number of valid unlabeled patches:", len(contrastive_dataset))
print("train/val split sizes:", len(train_dataset), len(val_dataset))
print("batch size:", batch_size)
print("device:", device)
if torch.cuda.is_available():
    print("CUDA GPU name:", torch.cuda.get_device_name(0))
print("model parameter count:", count_trainable_parameters(model))
print("projection_dim:", projection_dim)
print("temperature:", temperature)
print("max_epochs:", max_epochs)
print("learning rate:", learning_rate)
print("encoder checkpoint path:", encoder_best_path)
print("full model checkpoint path:", full_model_best_path)

training_log, training_summary = train_contrastive_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    full_model_best_path=full_model_best_path,
    encoder_best_path=encoder_best_path,
    last_checkpoint_path=last_checkpoint_path,
    config=config,
    channel_means=channel_means,
    channel_stds=channel_stds,
    max_epochs=max_epochs,
    early_stopping_patience=early_stopping_patience,
    temperature=temperature,
    batch_size=batch_size,
    grad_clip_norm=gradient_clip_norm,
)
training_log.to_csv(training_log_path, index=False)
print("Training summary:", training_summary)
print("Training log saved:", training_log_path)


Quality check before training
number of raster files: 14
raster shape: (10071, 9596)
patch_size: 32
number of valid unlabeled patches: 20000
train/val split sizes: 18000 2000
batch size: 128
device: cuda
CUDA GPU name: NVIDIA GeForce RTX 4090
model parameter count: 11504512
projection_dim: 128
temperature: 0.2
max_epochs: 50
learning rate: 0.0001
encoder checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\contrastive\resnet18_contrastive_ps32_encoder_best.pt
full model checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\contrastive\resnet18_contrastive_ps32_full_model_best.pt


epoch 001: train_loss=1.757970, val_loss=1.364925


epoch 002: train_loss=1.258451, val_loss=1.254798


epoch 003: train_loss=1.194147, val_loss=1.222041


epoch 004: train_loss=1.165595, val_loss=1.191606


...

epoch 047: train_loss=1.078331, val_loss=1.116506


epoch 048: train_loss=1.077859, val_loss=1.120471


epoch 049: train_loss=1.077904, val_loss=1.118875


epoch 050: train_loss=1.076249, val_loss=1.116099


Training summary: {'best_epoch': 45, 'best_val_loss': 1.1136543502807617, 'best_train_loss': 1.0784430215623644}
Training log saved: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\SSL_contrastive_ps32\training_logs\contrastive_ps32_training_log.csv


## 11. Save best pretrained encoder and logs


In [11]:
required_log_columns = ["epoch", "train_loss", "val_loss", "learning_rate", "temperature", "batch_size"]
missing_log_columns = [column for column in required_log_columns if column not in training_log.columns]
if missing_log_columns:
    raise ValueError(f"Training log missing columns: {missing_log_columns}")
for path in [training_log_path, encoder_best_path, full_model_best_path, last_checkpoint_path]:
    if not path.exists():
        raise FileNotFoundError(path)
encoder_checkpoint = torch.load(encoder_best_path, map_location="cpu")
full_checkpoint = torch.load(full_model_best_path, map_location="cpu")
print("Encoder checkpoint keys:", sorted(encoder_checkpoint.keys()))
print("Full model checkpoint keys:", sorted(full_checkpoint.keys()))
print("Encoder checkpoint epoch:", encoder_checkpoint["epoch"])
print("Encoder checkpoint val_loss:", encoder_checkpoint["val_loss"])


Encoder checkpoint keys: ['channel_means', 'channel_stds', 'config', 'encoder_state_dict', 'epoch', 'val_loss']
Full model checkpoint keys: ['channel_means', 'channel_stds', 'config', 'epoch', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'val_loss']
Encoder checkpoint epoch: 45
Encoder checkpoint val_loss: 1.1136543502807617


## 12. Plot contrastive training curves


In [12]:
plot_ssl_loss_curve(training_log, loss_curve_path)
if not loss_curve_path.exists():
    raise FileNotFoundError(loss_curve_path)
print("Loss curve saved:", loss_curve_path)


Loss curve saved: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_contrastive_ps32\training_curves\contrastive_ps32_loss_curve.png


## 13. Print final summary and next-step notes


In [13]:
final_train_loss = float(training_log["train_loss"].iloc[-1])
final_val_loss = float(training_log["val_loss"].iloc[-1])
print("Contrastive SSL pretraining summary")
print("number of valid unlabeled patches:", len(contrastive_dataset))
print("best epoch:", training_summary["best_epoch"])
print("best validation contrastive loss:", training_summary["best_val_loss"])
print("final train loss:", final_train_loss)
print("final val loss:", final_val_loss)
print("encoder checkpoint path:", encoder_best_path)
print("full model checkpoint path:", full_model_best_path)
print("last checkpoint path:", last_checkpoint_path)
print("training log path:", training_log_path)
print("loss curve path:", loss_curve_path)
print("Next step: use the contrastive encoder checkpoint in a supervised SCV fine-tuning notebook.")
contrastive_dataset.close()


Contrastive SSL pretraining summary
number of valid unlabeled patches: 20000
best epoch: 45
best validation contrastive loss: 1.1136543502807617
final train loss: 1.0762493708928427
final val loss: 1.1160991249084473
encoder checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\contrastive\resnet18_contrastive_ps32_encoder_best.pt
full model checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\contrastive\resnet18_contrastive_ps32_full_model_best.pt
last checkpoint path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\checkpoints\ssl_pretrained\contrastive\resnet18_contrastive_ps32_last.pt
training log path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\outputs\SSL_contrastive_ps32\training_logs\contrastive_ps32_training_log.csv
loss curve path: D:\SBU first year\First year paper\LSM_SSL_ResNet18_Project\figures\SSL_contrastive_ps32\training_curves\contrastive_

## 14. Visualize contrastive augmentation examples

Contrastive pretraining does not produce reconstruction outputs. Instead, this section visualizes the original normalized patch and two independently augmented views used by the SimCLR-style objective.

In [ ]:
from src.ssl_masked_recon import MaskedReconstructionRasterDataset
from src.plotting import plot_contrastive_augmentation_examples

augmentation_example_dir = PROJECT_ROOT / "figures" / "SSL_contrastive_ps32" / "augmentation_examples"
augmentation_example_path = augmentation_example_dir / "contrastive_augmented_views_examples_ps32.png"
ensure_dir(augmentation_example_dir)

# Reload the best checkpoint so this cell can be run independently after training.
encoder_checkpoint_for_examples = torch.load(encoder_best_path, map_location="cpu")
example_channel_means = encoder_checkpoint_for_examples["channel_means"]
example_channel_stds = encoder_checkpoint_for_examples["channel_stds"]
example_augmentation_config = encoder_checkpoint_for_examples.get("config", {}).get("augmentation_config", augmentation_config)

set_global_seed(random_seed)
original_example_dataset = MaskedReconstructionRasterDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    patch_size=patch_size,
    nodata_value=nodata_value,
    normalize=True,
    channel_means=example_channel_means,
    channel_stds=example_channel_stds,
    return_metadata=False,
)
contrastive_example_dataset = ContrastiveRasterPatchDataset(
    patch_index_csv=unlabeled_index_csv,
    raster_dir=raster_dir,
    patch_size=patch_size,
    nodata_value=nodata_value,
    normalize=True,
    channel_means=example_channel_means,
    channel_stds=example_channel_stds,
    return_metadata=False,
    augment=True,
    augmentation_config=example_augmentation_config,
)

X_example = original_example_dataset[0].unsqueeze(0)
view1_example, view2_example = contrastive_example_dataset[0]
view1_example = view1_example.unsqueeze(0)
view2_example = view2_example.unsqueeze(0)

plot_contrastive_augmentation_examples(
    X_example,
    view1_example,
    view2_example,
    output_path=augmentation_example_path,
    channel_indices=[0, 1, 2],
    channel_names=["factor_01", "factor_02", "factor_03"],
)
original_example_dataset.close()
contrastive_example_dataset.close()
print("Contrastive augmentation examples saved:", augmentation_example_path)